In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from src.model_loader import load_model, get_device
from src.hooks import extract_activations

In [11]:
model, tokenizer, device = load_model(
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
)

df = pd.read_csv("../results/cleaned_harmful_harmless_responses_labeled.csv")


refused_harmful = df[(df["type"] == "harmful") & (df["refused"] == True)]
harmless = df[df["type"] == "harmless"]

refused_prompts = refused_harmful["prompt"].tolist()
harmless_prompts = harmless["prompt"].tolist()

print(f"Refused harmful prompts: {len(refused_prompts)}")
print(f"Harmless prompts: {len(harmless_prompts)}")

complied_harmful = df[(df["type"] == "harmful") & (df["refused"] == False)]
complied_prompts = complied_harmful["prompt"].tolist()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B
Refused harmful prompts: 35
Harmless prompts: 50


In [12]:
LAYER = "model.layers.14"

print("Extracting refused harmful activations...")
refused_acts = extract_activations(model, tokenizer, refused_prompts, LAYER, device)

print("Extracting harmless activations...")
harmless_acts = extract_activations(model, tokenizer, harmless_prompts, LAYER, device)

print("Extracting complied harmful activations...")
complied_acts = extract_activations(model, tokenizer, complied_prompts, LAYER, device)

print(f"\nRefused acts shape: {refused_acts.shape}")
print(f"Harmless acts shape: {harmless_acts.shape}")

Extracting refused harmful activations...
Extracting harmless activations...
Extracting complied harmful activations...

Refused acts shape: (35, 2048)
Harmless acts shape: (50, 2048)


In [13]:
# Difference in means — Arditi et al. methodology
refusal_direction = refused_acts.mean(axis=0) - harmless_acts.mean(axis=0)
refusal_direction = refusal_direction / np.linalg.norm(refusal_direction)

print(f"Refusal direction shape: {refusal_direction.shape}")
print(f"Refusal direction norm: {np.linalg.norm(refusal_direction):.6f}")

# Save for use in ablation notebook
np.save("../data/refusal_direction_layer14.npy", refusal_direction)
print("Saved refusal direction.")

Refusal direction shape: (2048,)
Refusal direction norm: 1.000000
Saved refusal direction.


In [14]:
refused_proj = refused_acts @ refusal_direction
harmless_proj = harmless_acts @ refusal_direction
complied_proj = complied_acts @ refusal_direction

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=refused_proj, name="Refused harmful",
    opacity=0.65, marker_color="#ef4444", nbinsx=30,
    histnorm="probability density",
))
fig.add_trace(go.Histogram(
    x=harmless_proj, name="Harmless",
    opacity=0.65, marker_color="#3b82f6", nbinsx=30,
    histnorm="probability density",
))
fig.update_layout(
    barmode="overlay",
    title="Distribution of activations along refusal direction (Layer 14)",
    xaxis_title="Projection value",
    yaxis_title="Density",
    template="plotly_white",
    width=800, height=400,
)
fig.show()

In [15]:
all_proj = np.concatenate([refused_proj, harmless_proj])
bin_min, bin_max = float(all_proj.min()), float(all_proj.max())
bin_size = (bin_max - bin_min) / 30  # 30 bins across the full range

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=refused_proj, name="Refused harmful",
    opacity=0.65, marker_color="#ef4444",
    histnorm="probability density",
    xbins=dict(start=bin_min, end=bin_max, size=bin_size),
))
fig.add_trace(go.Histogram(
    x=harmless_proj, name="Harmless",
    opacity=0.65, marker_color="#3b82f6",
    histnorm="probability density",
    xbins=dict(start=bin_min, end=bin_max, size=bin_size),
))
fig.update_layout(
    barmode="overlay",
    title="Distribution of activations along refusal direction (Layer 14)",
    xaxis_title="Projection value",
    yaxis_title="Density",
    template="plotly_white",
    width=800, height=400,
)
fig.show()

In [16]:
import plotly.graph_objects as go
np.random.seed(42)  # reproducible jitter

refused_hover = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(refused_harmful.index, refused_harmful["prompt"], refused_proj)
]
complied_hover = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(complied_harmful.index, complied_harmful["prompt"], complied_proj)
]
harmless_hover = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(harmless.index, harmless["prompt"], harmless_proj)
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=refused_proj,
    y=np.random.uniform(0.6, 1.0, len(refused_proj)),
    mode="markers", name="Refused harmful",
    marker=dict(color="#ef4444", size=8, opacity=0.7),
    text=refused_hover,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=complied_proj,
    y=np.random.uniform(0.1, 0.5, len(complied_proj)),
    mode="markers", name="Complied harmful",
    marker=dict(color="#f97316", size=8, opacity=0.7),
    text=complied_hover,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=harmless_proj,
    y=np.random.uniform(-0.4, -0.1, len(harmless_proj)),
    mode="markers", name="Harmless",
    marker=dict(color="#3b82f6", size=8, opacity=0.7),
    text=harmless_hover,
    hovertemplate="%{text}<extra></extra>",
))

fig.update_layout(
    title="Projection onto refusal direction — hover to inspect prompts",
    xaxis_title="Projection value (← harmless | refused →)",
    yaxis=dict(
        visible=True,
        tickvals=[0.8, 0.3, -0.25],
        ticktext=["Refused harmful", "Complied harmful", "Harmless"],
    ),
    template="plotly_white",
    width=900, height=450,
    legend=dict(x=0.01, y=0.99),
)

fig.show()

In [17]:
# All refused prompts
refused_prompts = refused_harmful["prompt"].tolist()
refused_acts_all = refused_acts  # already computed

# All complied prompts — harmful compliances + harmless
complied_all = pd.concat([complied_harmful, harmless])
complied_prompts_all = complied_all["prompt"].tolist()

print("Extracting all complied activations...")
complied_acts_all = extract_activations(
    model, tokenizer, complied_prompts_all, LAYER, device
)

# Recompute direction — refused vs complied regardless of harmfulness
refusal_direction_v2 = refused_acts.mean(axis=0) - complied_acts_all.mean(axis=0)
refusal_direction_v2 = refusal_direction_v2 / np.linalg.norm(refusal_direction_v2)

np.save("../data/refusal_direction_v2_layer14.npy", refusal_direction_v2)
print(f"Direction shape: {refusal_direction_v2.shape}")

Extracting all complied activations...
Direction shape: (2048,)


In [18]:
# Project all groups onto new direction
refused_proj_v2 = refused_acts @ refusal_direction_v2
complied_proj_v2 = complied_acts @ refusal_direction_v2
harmless_proj_v2 = harmless_acts @ refusal_direction_v2

np.random.seed(42)

refused_hover_v2 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(refused_harmful.index, refused_harmful["prompt"], refused_proj_v2)
]
complied_hover_v2 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(complied_harmful.index, complied_harmful["prompt"], complied_proj_v2)
]
harmless_hover_v2 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(harmless.index, harmless["prompt"], harmless_proj_v2)
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=refused_proj_v2,
    y=np.random.uniform(0.6, 1.0, len(refused_proj_v2)),
    mode="markers", name="Refused harmful",
    marker=dict(color="#ef4444", size=8, opacity=0.7),
    text=refused_hover_v2,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=complied_proj_v2,
    y=np.random.uniform(0.1, 0.5, len(complied_proj_v2)),
    mode="markers", name="Complied harmful",
    marker=dict(color="#f97316", size=8, opacity=0.7),
    text=complied_hover_v2,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=harmless_proj_v2,
    y=np.random.uniform(-0.4, -0.1, len(harmless_proj_v2)),
    mode="markers", name="Harmless",
    marker=dict(color="#3b82f6", size=8, opacity=0.7),
    text=harmless_hover_v2,
    hovertemplate="%{text}<extra></extra>",
))

fig.update_layout(
    title="Projection onto refusal direction v2 (refused vs complied) — hover to inspect",
    xaxis_title="Projection value (← complied | refused →)",
    yaxis=dict(
        visible=True,
        tickvals=[0.8, 0.3, -0.25],
        ticktext=["Refused harmful", "Complied harmful", "Harmless"],
    ),
    template="plotly_white",
    width=900, height=450,
    legend=dict(x=0.01, y=0.99),
)

fig.show()

In [19]:
# v3 — pure refusal direction within harmful prompts only
# Both groups are harmful, only the refusal decision varies
refusal_direction_v3 = refused_acts.mean(axis=0) - complied_acts.mean(axis=0)
refusal_direction_v3 = refusal_direction_v3 / np.linalg.norm(refusal_direction_v3)

np.save("../data/refusal_direction_v3_layer14.npy", refusal_direction_v3)
print(f"Direction shape: {refusal_direction_v3.shape}")
print(f"Direction norm: {np.linalg.norm(refusal_direction_v3):.6f}")

# Project all three groups
refused_proj_v3 = refused_acts @ refusal_direction_v3
complied_proj_v3 = complied_acts @ refusal_direction_v3
harmless_proj_v3 = harmless_acts @ refusal_direction_v3

np.random.seed(42)

refused_hover_v3 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(refused_harmful.index, refused_harmful["prompt"], refused_proj_v3)
]
complied_hover_v3 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(complied_harmful.index, complied_harmful["prompt"], complied_proj_v3)
]
harmless_hover_v3 = [
    f"<b>Row {idx}</b><br>{prompt[:80]}...<br>Projection: {proj:.3f}"
    for idx, prompt, proj in zip(harmless.index, harmless["prompt"], harmless_proj_v3)
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=refused_proj_v3,
    y=np.random.uniform(0.6, 1.0, len(refused_proj_v3)),
    mode="markers", name="Refused harmful",
    marker=dict(color="#ef4444", size=8, opacity=0.7),
    text=refused_hover_v3,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=complied_proj_v3,
    y=np.random.uniform(0.1, 0.5, len(complied_proj_v3)),
    mode="markers", name="Complied harmful",
    marker=dict(color="#f97316", size=8, opacity=0.7),
    text=complied_hover_v3,
    hovertemplate="%{text}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=harmless_proj_v3,
    y=np.random.uniform(-0.4, -0.1, len(harmless_proj_v3)),
    mode="markers", name="Harmless",
    marker=dict(color="#3b82f6", size=8, opacity=0.7),
    text=harmless_hover_v3,
    hovertemplate="%{text}<extra></extra>",
))

fig.update_layout(
    title="Projection onto refusal direction v3 (refused vs complied harmful only) — hover to inspect",
    xaxis_title="Projection value (← complied | refused →)",
    yaxis=dict(
        visible=True,
        tickvals=[0.8, 0.3, -0.25],
        ticktext=["Refused harmful", "Complied harmful", "Harmless"],
    ),
    template="plotly_white",
    width=900, height=450,
    legend=dict(x=0.01, y=0.99),
)

fig.show()

Direction shape: (2048,)
Direction norm: 1.000000
